## Setup Demo Table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.final.bronze_customer_tt_demo
AS
SELECT *
FROM workspace.final.bronze_customer;

### Verify Record Count

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.final.bronze_customer_tt_demo;

### Delta History

In [0]:
%sql
DESCRIBE HISTORY workspace.final.bronze_customer_tt_demo;

### Create Version

In [0]:
%sql
UPDATE workspace.final.bronze_customer_tt_demo
SET TerritoryID = 999
WHERE CustomerID = 1;

In [0]:
%sql
UPDATE workspace.final.bronze_customer_tt_demo
SET TerritoryID = 888
WHERE CustomerID = 1;

In [0]:
%sql
DESCRIBE HISTORY workspace.final.bronze_customer_tt_demo;

### Time Travel

#### Version 0:

In [0]:
%sql
SELECT CustomerID, TerritoryID
FROM workspace.final.bronze_customer_tt_demo VERSION AS OF 0
WHERE CustomerID = 1;

#### Version 1:

In [0]:
%sql
SELECT CustomerID, TerritoryID
FROM workspace.final.bronze_customer_tt_demo VERSION AS OF 1
WHERE CustomerID = 1;

#### Version 2:

In [0]:
%sql
SELECT CustomerID, TerritoryID
FROM workspace.final.bronze_customer_tt_demo VERSION AS OF 2
WHERE CustomerID = 1;

### Schema Evolution

In [0]:
from pyspark.sql.functions import lit

new_customer_df = (
    spark.read.table(
        "workspace.final.bronze_customer"
    )
    .withColumn(
        "customer_segment",
        lit("PREMIUM")
    )
)

### Add New Column

In [0]:
(new_customer_df.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable(
     "workspace.final.bronze_customer_schema_demo"
 ))

### Write with Schema Evolution

In [0]:
%sql
DESCRIBE workspace.final.bronze_customer_schema_demo;

### Verify New Schema

### Schema Enforcement

In [0]:
bad_data = [
    (
        "ABC","XYZ","TEST","INVALID","ACCOUNT001","GUID001","BAD_DATE"
    )
]

bad_df = spark.createDataFrame(
    bad_data,
    [
        "CustomerID","PersonID","StoreID","TerritoryID","AccountNumber","rowguid","ModifiedDate"
    ]
)
display(bad_df)

### Create Invalid Data

In [0]:
try:

    (
        bad_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "workspace.final.bronze_customer_schema_demo"
        )
    )

    print("Load Successful")

except Exception as e:

    print("Schema Validation Failed")
    print(e)